In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from datetime import datetime
from scipy.stats import norm

np.random.seed(42)

#  PATHS AND RUN CONFIGURATION

SPLIT_DIR = Path("../backend/data/splits_70_15_15")
TRAIN_PATH = SPLIT_DIR / "train.parquet"

# Model artifacts required for comparison.
# - eval_hives_csv: hive IDs used for tuning/selection (kept for auditing)
# - steps_parquet : per-timestep predictions and targets (val/test)
# - has_nis       : whether NIS columns are available

MODELS = {
    "KF": {
        # Saved in: ..\backend\data\KF_outputs
        "eval_hives_csv": Path("../backend/data/KF_outputs/kalman filter_eval_hives_val_only.csv"),
        "steps_parquet":  Path("../backend/data/KF_outputs/kalman filter_val_test_forecast_nis.parquet"),
        "has_nis": True,
    },
    "EKF": {
        # Saved in: ..\backend\data\EKF_Outputs
        "eval_hives_csv": Path("../backend/data/EKF_Outputs/EKF_eval_hives_val_only.csv"),
        "steps_parquet":  Path("../backend/data/EKF_Outputs/EKF_val_test_forecast_nis.parquet"),
        "has_nis": True,
    },
    "EnKF": {
        # Saved in: ..\backend\data\Enkf_outputs
        "eval_hives_csv": Path("../backend/data/Enkf_outputs/EnKF_eval_hives_val_only.csv"),
        "steps_parquet":  Path("../backend/data/Enkf_outputs/EnKF_val_test_forecast_nis.parquet"),
        "has_nis": True,
    },
    "AdaptiveR": {
        # Saved in: ..\backend\data\AdaptiveR_KF
        "eval_hives_csv": Path("../backend/data/AdaptiveR_KF/kf_eval_hives_AdaptiveR_KF.csv"),
        "steps_parquet":  Path("../backend/data/AdaptiveR_KF/kf_val_test_forecast_nis_AdaptiveR_KF.parquet"),
        "has_nis": True,
    },
    "ARIMA": {
        # Keep your ARIMA paths as they were (based on your earlier structure)
        "eval_hives_csv": Path("../backend/data/arima_outputs_fair/arima_fair_eval_hives.csv"),
        "steps_parquet":  Path("../backend/data/arima_outputs_fair/arima_fair_val_test_forecast.parquet"),
        "has_nis": False,  
    },
}


RUN_TAG = datetime.now().strftime("%Y%m%d_%H%M%S")
OUT_DIR = Path("../backend/data/model_comparison") / f"run_{RUN_TAG}"
FIG_DIR = OUT_DIR / "figures"
TAB_DIR = OUT_DIR / "tables"

OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
TAB_DIR.mkdir(parents=True, exist_ok=True)

# Observations and corresponding step file column conventions
OBS_COLS = ["temperature", "humidity", "audio_density"]
TRUE_COLS = [f"y_true_{c}" for c in OBS_COLS]
PRED_COLS = [f"y_pred_{c}" for c in OBS_COLS]
STD_COLS = [f"pred_std_{c}" for c in OBS_COLS]  

# Fairness constraint- require at least this many comparable timesteps per hive per split.
MIN_ROWS_PER_SPLIT = 200

#  PLOTTING STYLE AND HELPERS

plt.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 600,
    "font.size": 12,
    "axes.titlesize": 16,
    "axes.labelsize": 14,
    "legend.fontsize": 11,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
})

def savefig_both(fig, stem: str):
    """Save the current figure as both PNG and PDF into FIG_DIR."""
    png_path = FIG_DIR / f"{stem}.png"
    pdf_path = FIG_DIR / f"{stem}.pdf"
    fig.savefig(png_path, bbox_inches="tight")
    fig.savefig(pdf_path, bbox_inches="tight")
    print("Saved:", png_path)
    print("Saved:", pdf_path)

def plot_representative_hive(hive_id: int, start=None, end=None, split_name="test"):
    """
    Plot a qualitative comparison for a single hive and split, showing:
      - Observed series
      - KF forecast mean and 95% prediction interval
      - AdaptiveR forecast mean and 95% prediction interval

    Notes:
      - This function aligns KF and AdaptiveR by timestamp before plotting.
      - Missing observations (NaNs) will naturally produce gaps in the plotted series.
    """
    kf = steps_cache["KF"]
    ar = steps_cache["AdaptiveR"]

    kf_h = kf[(kf["split"] == split_name) & (kf["hive_id"].astype(int) == hive_id)].copy()
    ar_h = ar[(ar["split"] == split_name) & (ar["hive_id"].astype(int) == hive_id)].copy()

    # Align on timestamp to compare the same timesteps
    df = kf_h.merge(ar_h, on=["published_at", "hive_id", "split"], suffixes=("_kf", "_ar"))

    if start is not None:
        df = df[df["published_at"] >= pd.to_datetime(start, utc=True)]
    if end is not None:
        df = df[df["published_at"] <= pd.to_datetime(end, utc=True)]

    if df.empty:
        print("No data in selected window for hive", hive_id)
        return

    fig = plt.figure(figsize=(12, 8))
    for i, col in enumerate(OBS_COLS, start=1):
        ax = plt.subplot(3, 1, i)

        t = df["published_at"]
        y = df[f"y_true_{col}_kf"]
        kf_mu = df[f"y_pred_{col}_kf"]
        kf_sd = df[f"pred_std_{col}_kf"]
        ar_mu = df[f"y_pred_{col}_ar"]
        ar_sd = df[f"pred_std_{col}_ar"]

        # Observations
        ax.plot(t, y, linewidth=1.2, label="Observed")

        # KF prediction band
        ax.plot(t, kf_mu, linewidth=1.1, label="KF mean")
        ax.fill_between(t, (kf_mu - 1.96 * kf_sd), (kf_mu + 1.96 * kf_sd), alpha=0.2, label="KF 95% PI")

        # AdaptiveR prediction band
        ax.plot(t, ar_mu, linewidth=1.1, label="AdaptiveR mean")
        ax.fill_between(t, (ar_mu - 1.96 * ar_sd), (ar_mu + 1.96 * ar_sd), alpha=0.2, label="AdaptiveR 95% PI")

        ax.set_ylabel(col)
        ax.grid(True, alpha=0.3)
        if i == 1:
            ax.set_title(f"Representative hive {hive_id} ({split_name})")
        if i == 3:
            ax.set_xlabel("Time")

    # Single legend for the entire figure
    handles, labels = plt.gca().get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", ncol=3)
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    savefig_both(fig, f"FigRepHive_{hive_id}_{split_name}")
    plt.close(fig)

#  DATA LOADING AND METRICS UTILITIES

def load_eval_hives(path: Path) -> list[int]:
    """
    Load a hive list from CSV.
    Accepts either 'hive_id' or a single unnamed/first column.
    """
    df = pd.read_csv(path)
    col = "hive_id" if "hive_id" in df.columns else df.columns[0]
    h = pd.to_numeric(df[col], errors="coerce").dropna().astype(int).tolist()
    return sorted(h)

def _safe_series(df: pd.DataFrame, col: str, default="") -> pd.Series:
    """Return df[col] if present; otherwise return a default-valued Series of matching length."""
    if col in df.columns:
        return df[col]
    return pd.Series([default] * len(df), index=df.index)

def clean_steps(df: pd.DataFrame) -> pd.DataFrame:
    """
    Standardize a per-timestep steps DataFrame across models.

    Ensures the output contains:
      - published_at (UTC datetime; derived from 'published_at' or 'timestamp')
      - hive_id (Int64; derived from 'hive_id' or 'tag_number')
      - split in {"val","test"}
      - y_true_* and y_pred_* numeric where present
      - pred_std_* numeric where present
      - NIS columns numeric where present
    """
    df = df.copy()

    # Timestamp normalization
    if "published_at" in df.columns:
        df["published_at"] = pd.to_datetime(df["published_at"], utc=True, errors="coerce")
    elif "timestamp" in df.columns:
        df["published_at"] = pd.to_datetime(df["timestamp"], utc=True, errors="coerce")
    else:
        df["published_at"] = pd.NaT

    # Hive identifier normalization
    if "hive_id" in df.columns:
        df["hive_id"] = pd.to_numeric(df["hive_id"], errors="coerce").astype("Int64")
    elif "tag_number" in df.columns:
        df["hive_id"] = pd.to_numeric(df["tag_number"], errors="coerce").astype("Int64")
    else:
        df["hive_id"] = pd.Series([pd.NA] * len(df), dtype="Int64")

    # Split normalization
    split_s = _safe_series(df, "split", default="")
    df["split"] = split_s.astype(str).str.strip().str.lower()

    # Numeric conversions for target/prediction columns
    for c in TRUE_COLS + PRED_COLS:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    # Prediction standard deviation columns (when available)
    for c in STD_COLS:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    # NIS columns (when available)
    if "nis_raw" in df.columns:
        df["nis_raw"] = pd.to_numeric(df["nis_raw"], errors="coerce")
    if "nis_norm" in df.columns:
        df["nis_norm"] = pd.to_numeric(df["nis_norm"], errors="coerce")
    if "nis_dof" in df.columns:
        df["nis_dof"] = pd.to_numeric(df["nis_dof"], errors="coerce").fillna(0).astype(int)

    # Keep only val/test rows with valid hive IDs
    df = df.dropna(subset=["hive_id"])
    df = df[df["split"].isin(["val", "test"])].copy()

    # Sorting
    if df["published_at"].notna().any():
        df = df.sort_values(["hive_id", "published_at"])
    else:
        df = df.sort_values(["hive_id"])

    return df.reset_index(drop=True)

def train_std_from_train(train_path: Path) -> np.ndarray:
    """
    Compute per-channel standard deviations from the TRAIN split.
    Used to normalize RMSE into zRMSE-like units (prevents leakage).
    """
    train_df = pd.read_parquet(train_path).copy()
    for c in OBS_COLS:
        train_df[c] = pd.to_numeric(train_df[c], errors="coerce")
    std = train_df[OBS_COLS].astype(float).std(skipna=True).to_numpy()
    std = np.where(np.isfinite(std) & (std > 1e-12), std, 1.0)
    return std

def persistence_baseline_missing_safe(z: np.ndarray) -> np.ndarray:
    """Persistence baseline: previous observed value per dimension, robust to missing values."""
    T, d = z.shape
    out = np.full((T, d), np.nan, float)
    last = np.full((d,), np.nan, float)
    for t in range(T):
        out[t] = last
        m = np.isfinite(z[t])
        last[m] = z[t, m]
    return out

def rmse_per_dim(y: np.ndarray, yhat: np.ndarray) -> np.ndarray:
    """Per-dimension RMSE, ignoring NaNs."""
    y = np.asarray(y, float)
    yhat = np.asarray(yhat, float)
    out = np.full((y.shape[1],), np.nan, float)
    for d in range(y.shape[1]):
        m = np.isfinite(y[:, d]) & np.isfinite(yhat[:, d])
        if m.sum() > 0:
            out[d] = float(np.sqrt(np.mean((y[m, d] - yhat[m, d]) ** 2)))
    return out

def agg_zrmse(rmse_raw: np.ndarray, train_std: np.ndarray) -> float:
    """Aggregate normalized RMSE across channels."""
    rmse_raw = np.asarray(rmse_raw, float)
    m = np.isfinite(rmse_raw) & np.isfinite(train_std) & (train_std > 1e-12)
    if m.sum() == 0:
        return np.nan
    return float(np.mean(rmse_raw[m] / train_std[m]))

def bootstrap_ci(x, stat_fn=np.nanmedian, n_boot=2000, alpha=0.05, seed=42):
    """
    Non-parametric bootstrap confidence interval for a statistic (default: median).
    Returns: (statistic, CI_low, CI_high).
    """
    rng = np.random.default_rng(seed)
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    if x.size < 5:
        return (np.nan, np.nan, np.nan)
    boots = []
    for _ in range(n_boot):
        s = rng.choice(x, size=x.size, replace=True)
        boots.append(stat_fn(s))
    boots = np.array(boots, float)
    lo = np.nanpercentile(boots, 100 * (alpha / 2))
    hi = np.nanpercentile(boots, 100 * (1 - alpha / 2))
    return (stat_fn(x), lo, hi)

def cliff_delta(x, y):
    """
    Cliff's delta effect size in [-1, 1].
    Negative values typically indicate x < y.
    """
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    x = x[np.isfinite(x)]
    y = y[np.isfinite(y)]
    if x.size == 0 or y.size == 0:
        return np.nan
    # O(n^2) 
    gt = 0
    lt = 0
    for a in x:
        gt += np.sum(a > y)
        lt += np.sum(a < y)
    return (gt - lt) / (x.size * y.size)

def holm_adjust(pvals: pd.Series) -> pd.Series:
    """
    Holm–Bonferroni step-down adjustment for multiple hypotheses.
    """
    p = pvals.values.astype(float)
    n = len(p)
    order = np.argsort(p)
    adj = np.empty(n, float)
    for i, idx in enumerate(order):
        adj[idx] = (n - i) * p[idx]
    # Enforce monotonicity
    adj_sorted = adj[order]
    adj_sorted = np.maximum.accumulate(adj_sorted[::-1])[::-1]
    adj[order] = np.clip(adj_sorted, 0, 1)
    return pd.Series(adj, index=pvals.index)

def per_hive_channel_metrics(steps: pd.DataFrame, split_name: str, hives: list[int], train_std: np.ndarray):
    """
    Compute per-hive per-channel normalized RMSE (nRMSE) and aggregate nRMSE for a single model.

    Returns a DataFrame with one row per hive containing:
      - agg_nrmse
      - baseline_agg_nrmse
      - improvement_pct
      - per-channel nrmse and baseline nrmse
    """
    df_s = steps[(steps["split"] == split_name) & (steps["hive_id"].astype(int).isin(hives))].copy()
    rows = []
    for hive_id, g in df_s.groupby("hive_id"):
        y = g[TRUE_COLS].to_numpy(float)
        yhat = g[PRED_COLS].to_numpy(float)
        base = persistence_baseline_missing_safe(y)

        rmse_m = rmse_per_dim(y, yhat)
        rmse_b = rmse_per_dim(y, base)

        nrmse_m = rmse_m / train_std
        nrmse_b = rmse_b / train_std

        agg_m = np.nanmean(nrmse_m)
        agg_b = np.nanmean(nrmse_b)

        impr = np.nan
        if np.isfinite(agg_b) and agg_b > 0 and np.isfinite(agg_m):
            impr = 100.0 * (agg_b - agg_m) / agg_b

        row = {
            "hive_id": int(hive_id),
            "agg_nrmse": float(agg_m),
            "baseline_agg_nrmse": float(agg_b),
            "improvement_pct": float(impr) if np.isfinite(impr) else np.nan,
        }
        for i, col in enumerate(OBS_COLS):
            row[f"nrmse_{col}"] = float(nrmse_m[i]) if np.isfinite(nrmse_m[i]) else np.nan
            row[f"baseline_nrmse_{col}"] = float(nrmse_b[i]) if np.isfinite(nrmse_b[i]) else np.nan
        rows.append(row)
    return pd.DataFrame(rows)


# TRAIN STANDARD DEVIATIONS (FOR NORMALIZATION)

TRAIN_STD = train_std_from_train(TRAIN_PATH)
print("TRAIN std:", dict(zip(OBS_COLS, TRAIN_STD)))


# LOAD PER-TIMESTEP STEPS FOR ALL MODELS

steps_cache = {}
for model_name, cfg in MODELS.items():
    print(f"Loading {model_name} steps...")
    steps_cache[model_name] = clean_steps(pd.read_parquet(cfg["steps_parquet"]))


# FAIRNESS: BUILD COMMON HIVE SETS BASED ON ACTUAL STEP AVAILABILITY

def usable_hives_from_steps(df_steps: pd.DataFrame, split: str) -> set[int]:
    """
    Determine which hives have enough comparable timesteps for a given split.
    A timestep is counted as comparable if at least one dimension has finite
    y_true and y_pred.
    """
    df_s = df_steps[df_steps["split"] == split].copy()
    usable = set()
    for hive_id, g in df_s.groupby("hive_id"):
        z = g[TRUE_COLS].to_numpy(float)
        p = g[PRED_COLS].to_numpy(float)
        valid = np.isfinite(z).any(axis=1) & np.isfinite(p).any(axis=1)
        if valid.sum() >= MIN_ROWS_PER_SPLIT:
            usable.add(int(hive_id))
    return usable

usable_by_model = {"val": {}, "test": {}}
for split in ["val", "test"]:
    for model_name in MODELS:
        usable_by_model[split][model_name] = usable_hives_from_steps(steps_cache[model_name], split)

common_val = set.intersection(*[usable_by_model["val"][m] for m in MODELS])
common_test = set.intersection(*[usable_by_model["test"][m] for m in MODELS])

common_val = sorted(common_val)
common_test = sorted(common_test)

print("\nCommon hives (based on step availability):")
print("  val :", len(common_val))
print("  test:", len(common_test))

pd.DataFrame({"hive_id": common_val}).to_csv(OUT_DIR / "common_hives_val.csv", index=False)
pd.DataFrame({"hive_id": common_test}).to_csv(OUT_DIR / "common_hives_test.csv", index=False)

# Also save the intersection of the per-model eval_hives CSVs (auditing only)
eval_hives_by_model = {m: load_eval_hives(cfg["eval_hives_csv"]) for m, cfg in MODELS.items()}
common_eval_csv = sorted(list(set.intersection(*[set(v) for v in eval_hives_by_model.values()])))
pd.DataFrame({"hive_id": common_eval_csv}).to_csv(OUT_DIR / "eval_hives_master_from_csvs.csv", index=False)

def median_iqr(x):
    """Return (median, Q1, Q3) for a numeric array, ignoring NaNs."""
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    if x.size == 0:
        return (np.nan, np.nan, np.nan)
    return (np.nanmedian(x), np.nanpercentile(x, 25), np.nanpercentile(x, 75))


#  TABLE I: ACCURACY SUMMARY (MEDIAN AND IQR), FAIR COMMON HIVE SETS

acc_rows = []
for split_name, hives in [("val", common_val), ("test", common_test)]:
    for model_name in MODELS:
        steps = steps_cache[model_name]
        ph = per_hive_channel_metrics(steps, split_name, hives, TRAIN_STD)

        m_agg, q1_agg, q3_agg = median_iqr(ph["agg_nrmse"])
        m_imp, q1_imp, q3_imp = median_iqr(ph["improvement_pct"])

        row = {
            "split": split_name,
            "model": model_name,
            "n_hives": int(len(ph)),
            "agg_nrmse_median": m_agg,
            "agg_nrmse_q1": q1_agg,
            "agg_nrmse_q3": q3_agg,
            "improvement_median": m_imp,
            "improvement_q1": q1_imp,
            "improvement_q3": q3_imp,
        }
        for col in OBS_COLS:
            m, q1, q3 = median_iqr(ph[f"nrmse_{col}"])
            row[f"{col}_nrmse_median"] = m
            row[f"{col}_nrmse_q1"] = q1
            row[f"{col}_nrmse_q3"] = q3

        acc_rows.append(row)

tableI = pd.DataFrame(acc_rows)

def fmt_med_iqr(m, q1, q3, decimals=3):
    """Format as 'median [Q1, Q3]' for reporting."""
    if not np.isfinite(m):
        return ""
    return f"{m:.{decimals}f} [{q1:.{decimals}f}, {q3:.{decimals}f}]"

tableI_fmt = tableI.copy()
for col in OBS_COLS:
    tableI_fmt[f"{col}_nrmse"] = tableI.apply(
        lambda r: fmt_med_iqr(r[f"{col}_nrmse_median"], r[f"{col}_nrmse_q1"], r[f"{col}_nrmse_q3"]),
        axis=1,
    )

tableI_fmt["agg_nrmse"] = tableI.apply(
    lambda r: fmt_med_iqr(r["agg_nrmse_median"], r["agg_nrmse_q1"], r["agg_nrmse_q3"]),
    axis=1,
)
tableI_fmt["improvement_%"] = tableI.apply(
    lambda r: fmt_med_iqr(r["improvement_median"], r["improvement_q1"], r["improvement_q3"], decimals=1),
    axis=1,
)

tableI_fmt = tableI_fmt[[
    "split", "model", "n_hives",
    "temperature_nrmse", "humidity_nrmse", "audio_density_nrmse",
    "agg_nrmse", "improvement_%",
]]

tableI.to_csv(OUT_DIR / "TableI_accuracy_raw.csv", index=False)
tableI_fmt.to_csv(OUT_DIR / "TableI_accuracy_formatted.csv", index=False)
tableI_fmt.to_latex(OUT_DIR / "TableI_accuracy_formatted.tex", index=False)
print("Saved Table I outputs.")


# PER-HIVE METRICS (VAL + TEST), FAIR COMMON HIVE SETS

per_hive_rows = []

for model_name in MODELS:
    steps = steps_cache[model_name]

    for split_name, common_hives in [("val", common_val), ("test", common_test)]:
        df_s = steps[
            (steps["split"] == split_name) &
            (steps["hive_id"].astype(int).isin(common_hives))
        ].copy()

        for hive_id, g in df_s.groupby("hive_id"):
            z = g[TRUE_COLS].to_numpy(float)
            pred = g[PRED_COLS].to_numpy(float)

            base = persistence_baseline_missing_safe(z)
            rmse_base = rmse_per_dim(z, base)
            rmse_mod = rmse_per_dim(z, pred)

            base_agg = agg_zrmse(rmse_base, TRAIN_STD)
            mod_agg = agg_zrmse(rmse_mod, TRAIN_STD)

            impr = np.nan
            if np.isfinite(base_agg) and base_agg > 0 and np.isfinite(mod_agg):
                impr = 100.0 * (base_agg - mod_agg) / base_agg

            per_hive_rows.append({
                "model": model_name,
                "split": split_name,
                "hive_id": int(hive_id),
                "baseline_agg_zrmse": base_agg,
                "model_agg_zrmse": mod_agg,
                "improvement_pct": impr,
                "n_rows": int(len(g)),
            })

per_hive_df = (
    pd.DataFrame(per_hive_rows)
    .sort_values(["split", "model", "hive_id"])
    .reset_index(drop=True)
)
per_hive_df.to_csv(OUT_DIR / "per_hive_metrics_all_models.csv", index=False)

# Convenience view for test split downstream
test_df = per_hive_df[per_hive_df["split"] == "test"].copy()


#  BASELINE CONSISTENCY CHECK (TEST)

baseline_check = (
    test_df.groupby("hive_id")["baseline_agg_zrmse"]
    .agg(["min", "max", "mean", "count"])
    .reset_index()
)
baseline_check["range"] = baseline_check["max"] - baseline_check["min"]
baseline_check.to_csv(OUT_DIR / "baseline_consistency_check_test.csv", index=False)

max_range = float(baseline_check["range"].max()) if len(baseline_check) else np.nan
print("\nBaseline consistency check (test): max range across models =", max_range)


# FIGURES (USING FAIR TEST SET)

baseline_vals = test_df.groupby("hive_id")["baseline_agg_zrmse"].first().dropna().to_numpy()

# Figure 1a: log-scale boxplot including EnKF
labels = ["Baseline"]
data = [baseline_vals]
for m in ["KF", "EKF", "EnKF", "AdaptiveR", "ARIMA"]:
    data.append(test_df[test_df["model"] == m]["model_agg_zrmse"].dropna().to_numpy())
    labels.append(m)

fig = plt.figure(figsize=(10, 5))
plt.boxplot(data, labels=labels, showfliers=False)
plt.yscale("log")
plt.ylabel("Test agg_zRMSE (log scale)")
plt.title("Per-hive forecasting error distribution (Test split)")
plt.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
savefig_both(fig, "Figure1a_Test_Boxplot_LOG")
plt.close(fig)

# Figure 1b: zoomed boxplot excluding EnKF
labels2 = ["Baseline", "KF", "EKF", "AdaptiveR", "ARIMA"]
data2 = [
    baseline_vals,
    test_df[test_df["model"] == "KF"]["model_agg_zrmse"].dropna().to_numpy(),
    test_df[test_df["model"] == "EKF"]["model_agg_zrmse"].dropna().to_numpy(),
    test_df[test_df["model"] == "AdaptiveR"]["model_agg_zrmse"].dropna().to_numpy(),
    test_df[test_df["model"] == "ARIMA"]["model_agg_zrmse"].dropna().to_numpy(),
]

fig = plt.figure(figsize=(10, 5))
plt.boxplot(data2, labels=labels2, showfliers=False)
plt.ylabel("Test agg_zRMSE")
plt.title("Per-hive forecasting error distribution (Test split, zoom)")
plt.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
savefig_both(fig, "Figure1b_Test_Boxplot_ZOOM")
plt.close(fig)

# Figure A: improvement distribution relative to baseline
fig = plt.figure(figsize=(10, 5))
imp_data = []
imp_labels = []
for m in ["KF", "EKF", "AdaptiveR", "ARIMA", "EnKF"]:
    vals = test_df[test_df["model"] == m]["improvement_pct"].dropna().to_numpy()
    imp_data.append(vals)
    imp_labels.append(m)

plt.boxplot(imp_data, labels=imp_labels, showfliers=False)
plt.axhline(0, linestyle="--", linewidth=1.5)
plt.ylabel("Improvement vs baseline (%)  (positive = better)")
plt.title("Per-hive improvement over persistence baseline (Test split)")
plt.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
savefig_both(fig, "FigureA_ImprovementPct_Boxplot_Test")
plt.close(fig)

def paired_scatter(model_name: str):
    """
    Paired scatter plot: baseline error vs model error (per hive).
    Points below the diagonal indicate improvement over baseline.
    """
    t = test_df[test_df["model"] == model_name][["hive_id", "baseline_agg_zrmse", "model_agg_zrmse"]].dropna()
    if len(t) == 0:
        return
    x = t["baseline_agg_zrmse"].to_numpy()
    y = t["model_agg_zrmse"].to_numpy()
    mn = float(np.nanmin(np.r_[x, y]))
    mx = float(np.nanmax(np.r_[x, y]))

    fig = plt.figure(figsize=(5.5, 5.5))
    plt.scatter(x, y, alpha=0.75)
    plt.plot([mn, mx], [mn, mx], linestyle="--", linewidth=1.5)
    plt.xlabel("Baseline agg_zRMSE")
    plt.ylabel(f"{model_name} agg_zRMSE")
    plt.title(f"{model_name} vs Baseline (Test)  (below diagonal = better)")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    savefig_both(fig, f"FigureB_{model_name}_vs_Baseline_Scatter_Test")
    plt.close(fig)

paired_scatter("KF")
paired_scatter("AdaptiveR")

# Figure C: count of per-hive winners (lowest error)
idx = test_df.groupby("hive_id")["model_agg_zrmse"].idxmin()
winners = test_df.loc[idx, "model"].value_counts().reindex(list(MODELS.keys()), fill_value=0)

fig = plt.figure(figsize=(8, 4))
plt.bar(winners.index, winners.values)
plt.ylabel("Number of hives won")
plt.title("Per-hive best model count (Test split)")
plt.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
savefig_both(fig, "FigureC_PerHive_BestModel_Count_Test")
plt.close(fig)

winners.to_csv(OUT_DIR / "per_hive_best_model_count_test.csv")

# Figure D: median error ranking
ranking = (
    test_df.groupby("model")["model_agg_zrmse"]
    .median()
    .sort_values()
    .reindex(list(MODELS.keys()))
)

fig = plt.figure(figsize=(9, 4.5))
plt.bar(ranking.index, ranking.values)
plt.ylabel("Median Test agg_zRMSE (lower is better)")
plt.title("Model ranking by median error (Test split)")
plt.grid(True, axis="y", alpha=0.3)

for i, v in enumerate(ranking.values):
    if np.isfinite(v):
        plt.text(i, v, f"{v:.3f}", ha="center", va="bottom", fontsize=10)

plt.tight_layout()
savefig_both(fig, "FigureD_MedianError_Ranking_Test")
plt.close(fig)

(
    ranking.reset_index()
    .rename(columns={"index": "model", 0: "median_test_agg_zrmse"})
    .to_csv(OUT_DIR / "model_ranking_test.csv", index=False)
)

# Representative hive plot (qualitative)
if len(common_test) > 0:
    example_hive = common_test[0]
    plot_representative_hive(example_hive, split_name="test")


#  ADDITIONAL ANALYSES FOR REPORTING
#   A) Bootstrap confidence intervals
#   B) Pairwise Wilcoxon tests with Holm correction
#   C) Prediction interval coverage and sharpness (where pred_std is available)
#   D) NIS calibration against chi-square thresholds (DOF-aware)


#  Bootstrap confidence intervals 
summary_rows = []
for split_name in ["val", "test"]:
    df_s = per_hive_df[per_hive_df["split"] == split_name]
    for m in MODELS:
        vals = df_s[df_s["model"] == m]["model_agg_zrmse"].to_numpy()
        med, lo, hi = bootstrap_ci(vals, stat_fn=np.nanmedian, n_boot=3000)
        mean, lo2, hi2 = bootstrap_ci(vals, stat_fn=np.nanmean, n_boot=3000)
        summary_rows.append({
            "split": split_name,
            "model": m,
            "median_err": med,
            "median_err_ci_lo": lo,
            "median_err_ci_hi": hi,
            "mean_err": mean,
            "mean_err_ci_lo": lo2,
            "mean_err_ci_hi": hi2,
            "n_hives": int(np.isfinite(vals).sum()),
        })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(OUT_DIR / "summary_bootstrap_ci.csv", index=False)
summary_df.to_latex(OUT_DIR / "summary_bootstrap_ci.tex", index=False, float_format="%.4f")

# Pairwise Wilcoxon + Holm correction 
from scipy.stats import wilcoxon

pair_rows = []
dfT = test_df.copy()

# Hive-aligned matrix: rows = hive_id, cols = model, values = error
err_wide = dfT.pivot_table(index="hive_id", columns="model", values="model_agg_zrmse", aggfunc="first")
err_wide = err_wide.dropna()  # require all models present
err_wide.to_csv(OUT_DIR / "test_errors_wide_all_models.csv")

models_list = list(MODELS.keys())

for i in range(len(models_list)):
    for j in range(i + 1, len(models_list)):
        a = models_list[i]
        b = models_list[j]
        xa = err_wide[a].to_numpy()
        xb = err_wide[b].to_numpy()
        if len(xa) < 3:
            continue
        stat, p = wilcoxon(xa, xb)  # two-sided test
        diff = xa - xb
        pair_rows.append({
            "model_A": a,
            "model_B": b,
            "n_hives": int(len(diff)),
            "median_delta_A_minus_B": float(np.nanmedian(diff)),
            "win_rate_A_lt_B": float(np.mean(xa < xb)),
            "wilcoxon_stat": float(stat),
            "p_value": float(p),
            "cliffs_delta_A_vs_B": float(cliff_delta(xa, xb)),
        })

pair_df = pd.DataFrame(pair_rows)
if len(pair_df):
    pair_df["p_holm"] = holm_adjust(pair_df["p_value"])
pair_df.to_csv(OUT_DIR / "pairwise_wilcoxon_all_models_test.csv", index=False)
pair_df.to_latex(OUT_DIR / "pairwise_wilcoxon_all_models_test.tex", index=False, float_format="%.4g")

# Prediction interval coverage and sharpness 
# Gaussian approximation: y_pred ± z * pred_std
z80 = 1.2815515655446004   # norm.ppf(0.9): two-sided 80% interval
z95 = 1.959963984540054    # norm.ppf(0.975): two-sided 95% interval

cov_rows = []

for model_name in MODELS:
    steps = steps_cache[model_name]
    if not all(c in steps.columns for c in STD_COLS):
        continue

    for split_name, common_hives in [("val", common_val), ("test", common_test)]:
        df_s = steps[
            (steps["split"] == split_name) &
            (steps["hive_id"].astype(int).isin(common_hives))
        ].copy()
        if df_s.empty:
            continue

        for col in OBS_COLS:
            yt = pd.to_numeric(df_s[f"y_true_{col}"], errors="coerce").to_numpy(float)
            yp = pd.to_numeric(df_s[f"y_pred_{col}"], errors="coerce").to_numpy(float)
            ys = pd.to_numeric(df_s[f"pred_std_{col}"], errors="coerce").to_numpy(float)

            m = np.isfinite(yt) & np.isfinite(yp) & np.isfinite(ys) & (ys > 0)
            if m.sum() < 200:
                continue

            e = yt[m] - yp[m]
            abs_e = np.abs(e)

            cov80 = float(np.mean(abs_e <= z80 * ys[m]))
            cov95 = float(np.mean(abs_e <= z95 * ys[m]))
            sharp = float(np.mean(ys[m]))

            cov_rows.append({
                "model": model_name,
                "split": split_name,
                "dimension": col,
                "n_samples": int(m.sum()),
                "coverage_80": cov80,
                "coverage_95": cov95,
                "mean_pred_std": sharp,
            })

cov_df = pd.DataFrame(cov_rows)
cov_df.to_csv(OUT_DIR / "prediction_interval_coverage_and_sharpness.csv", index=False)

# Coverage plots (test split, averaged across dimensions)
if len(cov_df):
    cov_test = cov_df[cov_df["split"] == "test"].copy()
    cov_avg = cov_test.groupby("model")[["coverage_80", "coverage_95", "mean_pred_std"]].mean().reindex(models_list)
    cov_avg.to_csv(OUT_DIR / "coverage_summary_test.csv")

    fig = plt.figure(figsize=(8, 4))
    plt.bar(cov_avg.index, cov_avg["coverage_95"].to_numpy())
    plt.axhline(0.95, linestyle="--", linewidth=1.5)
    plt.ylabel("Empirical 95% coverage")
    plt.title("Uncertainty calibration: 95% prediction interval coverage (Test)")
    plt.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    savefig_both(fig, "FigCov1_PI95_Coverage_Test")
    plt.close(fig)

    fig = plt.figure(figsize=(8, 4))
    plt.bar(cov_avg.index, cov_avg["mean_pred_std"].to_numpy())
    plt.ylabel("Mean predicted std (sharpness)")
    plt.title("Uncertainty sharpness (Test)")
    plt.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    savefig_both(fig, "FigCov2_Sharpness_Test")
    plt.close(fig)

# Reliability curve data (multiple nominal levels)
levels = [0.50, 0.80, 0.90, 0.95]

rel_rows = []
for model_name in MODELS:
    steps = steps_cache[model_name]
    if not all(c in steps.columns for c in STD_COLS):
        continue

    for split_name, common_hives in [("val", common_val), ("test", common_test)]:
        df_s = steps[
            (steps["split"] == split_name) &
            (steps["hive_id"].astype(int).isin(common_hives))
        ].copy()
        if df_s.empty:
            continue

        for level in levels:
            z = norm.ppf((1 + level) / 2.0)
            covs = []

            for col in OBS_COLS:
                yt = pd.to_numeric(df_s[f"y_true_{col}"], errors="coerce").to_numpy(float)
                yp = pd.to_numeric(df_s[f"y_pred_{col}"], errors="coerce").to_numpy(float)
                ys = pd.to_numeric(df_s[f"pred_std_{col}"], errors="coerce").to_numpy(float)

                m = np.isfinite(yt) & np.isfinite(yp) & np.isfinite(ys) & (ys > 0)
                if m.sum() < 200:
                    continue

                cov = float(np.mean(np.abs(yt[m] - yp[m]) <= z * ys[m]))
                covs.append(cov)

            if len(covs) > 0:
                rel_rows.append({
                    "model": model_name,
                    "split": split_name,
                    "nominal": level,
                    "empirical": float(np.mean(covs)),
                })

rel_df = pd.DataFrame(rel_rows)
rel_df.to_csv(OUT_DIR / "reliability_curve_data.csv", index=False)

# Reliability plot (test split)
rel_test = rel_df[rel_df["split"] == "test"].copy()
if len(rel_test):
    fig = plt.figure(figsize=(6, 6))
    for model_name, g in rel_test.groupby("model"):
        g = g.sort_values("nominal")
        plt.plot(g["nominal"], g["empirical"], marker="o", label=model_name)

    plt.plot([0.5, 0.95], [0.5, 0.95], linestyle="--", linewidth=1.5)
    plt.xlabel("Nominal coverage")
    plt.ylabel("Empirical coverage")
    plt.title("Reliability of prediction intervals (Test)")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    savefig_both(fig, "FigRel_ReliabilityCurve_Test")
    plt.close(fig)

#  NIS calibration (chi-square exceed rates, DOF-aware) 
from scipy.stats import chi2

nis_models = [m for m in MODELS if MODELS[m]["has_nis"]]
nis_cal_rows = []

for model_name in nis_models:
    steps = steps_cache[model_name]
    for split_name, common_hives in [("val", common_val), ("test", common_test)]:
        df_s = steps[
            (steps["split"] == split_name) &
            (steps["hive_id"].astype(int).isin(common_hives))
        ].copy()
        if df_s.empty:
            continue

        if not all(c in df_s.columns for c in ["nis_raw", "nis_dof"]):
            continue

        m = (df_s["nis_dof"] > 0) & np.isfinite(df_s["nis_raw"])
        v = df_s.loc[m, ["nis_raw", "nis_dof"]].copy()
        if len(v) < 500:
            continue

        thr95 = chi2.ppf(0.95, v["nis_dof"].to_numpy())
        thr99 = chi2.ppf(0.99, v["nis_dof"].to_numpy())
        ex95 = float(np.mean(v["nis_raw"].to_numpy() > thr95))
        ex99 = float(np.mean(v["nis_raw"].to_numpy() > thr99))

        nis_cal_rows.append({
            "model": model_name,
            "split": split_name,
            "n_samples": int(len(v)),
            "chi2_exceed_rate_p95": ex95,  # expected ~0.05 under ideal calibration
            "chi2_exceed_rate_p99": ex99,  # expected ~0.01 under ideal calibration
        })

nis_cal_df = pd.DataFrame(nis_cal_rows)
nis_cal_df.to_csv(OUT_DIR / "nis_chi2_calibration_overall_by_model.csv", index=False)

# Additional NIS tail summaries using normalized NIS = nis_raw / dof
nis_tail_rows = []
for model_name in nis_models:
    steps = steps_cache[model_name]
    for split_name, common_hives in [("val", common_val), ("test", common_test)]:
        df_s = steps[
            (steps["split"] == split_name) &
            (steps["hive_id"].astype(int).isin(common_hives))
        ].copy()
        if df_s.empty or "nis_raw" not in df_s.columns or "nis_dof" not in df_s.columns:
            continue
        m = (df_s["nis_dof"] > 0) & np.isfinite(df_s["nis_raw"])
        v = df_s.loc[m, ["nis_raw", "nis_dof"]].to_numpy(float)
        if len(v) < 500:
            continue
        nis_norm = v[:, 0] / np.maximum(v[:, 1], 1.0)
        nis_tail_rows.append({
            "model": model_name,
            "split": split_name,
            "nis_norm_median": float(np.nanmedian(nis_norm)),
            "nis_norm_p95": float(np.nanpercentile(nis_norm, 95)),
            "nis_norm_p99": float(np.nanpercentile(nis_norm, 99)),
            "n_samples": int(len(nis_norm)),
        })

nis_tail_df = pd.DataFrame(nis_tail_rows)
nis_tail_df.to_csv(OUT_DIR / "nis_norm_tail_all_models.csv", index=False)
print("Saved NIS normalized tail summary for all NIS-capable models.")

# Table II: calibration summary (coverage + NIS calibration + NIS tail)
cov_avg = (
    cov_df.groupby(["model", "split"])[["coverage_80", "coverage_95", "mean_pred_std"]]
    .mean()
    .reset_index()
)
nis_ex = nis_cal_df.copy()
nis_tail = nis_tail_df.copy()

tableII = (
    cov_avg.merge(nis_ex, on=["model", "split"], how="left")
    .merge(nis_tail, on=["model", "split"], how="left")
)

tableII_fmt = tableII.copy()
for c in ["coverage_80", "coverage_95", "chi2_exceed_rate_p95", "chi2_exceed_rate_p99"]:
    if c in tableII_fmt.columns:
        tableII_fmt[c] = (tableII_fmt[c] * 100).round(1)

if "mean_pred_std" in tableII_fmt.columns:
    tableII_fmt["mean_pred_std"] = tableII_fmt["mean_pred_std"].round(3)

for c in ["nis_norm_median", "nis_norm_p95", "nis_norm_p99"]:
    if c in tableII_fmt.columns:
        tableII_fmt[c] = tableII_fmt[c].round(3)

cols = [
    "split", "model",
    "coverage_80", "coverage_95", "mean_pred_std",
    "chi2_exceed_rate_p95", "chi2_exceed_rate_p99",
    "nis_norm_median", "nis_norm_p95", "nis_norm_p99",
]
cols = [c for c in cols if c in tableII_fmt.columns]
tableII_fmt = tableII_fmt[cols]

tableII.to_csv(OUT_DIR / "TableII_calibration_raw.csv", index=False)
tableII_fmt.to_csv(OUT_DIR / "TableII_calibration_formatted.csv", index=False)
tableII_fmt.to_latex(OUT_DIR / "TableII_calibration_formatted.tex", index=False)
print("Saved Table II outputs.")

# Plot: chi-square exceed rate at 95% level (test split)
if len(nis_cal_df):
    t = nis_cal_df[nis_cal_df["split"] == "test"].set_index("model").reindex(nis_models)
    if len(t):
        fig = plt.figure(figsize=(8, 4))
        plt.bar(t.index, t["chi2_exceed_rate_p95"].to_numpy() * 100)
        plt.axhline(5.0, linestyle="--", linewidth=1.5)
        plt.ylabel("% above χ² 95% bound (raw NIS)")
        plt.title("NIS calibration (Test): exceed rate should be approximately 5%")
        plt.grid(True, axis="y", alpha=0.3)
        plt.tight_layout()
        savefig_both(fig, "FigNIS_Chi2_ExceedRate_p95_Test")
        plt.close(fig)

# NIS FIGURES: KF VS ADAPTIVER (DOF-AWARE THRESHOLDS)

print("\nGenerating NIS calibration summaries (KF vs AdaptiveR) with DOF-aware thresholds...")

nis_models_pair = ["KF", "AdaptiveR"]
nis_rows = []

for split_name, common_hives in [("val", common_val), ("test", common_test)]:
    for model_name in nis_models_pair:
        steps = steps_cache[model_name]
        df_s = steps[
            (steps["split"] == split_name) &
            (steps["hive_id"].astype(int).isin(common_hives))
        ].copy()

        if "nis_raw" not in df_s.columns or "nis_dof" not in df_s.columns:
            continue

        valid = df_s[(df_s["nis_dof"] > 0) & np.isfinite(df_s["nis_raw"]) & np.isfinite(df_s["nis_dof"])]
        if len(valid) == 0:
            continue

        raw = valid["nis_raw"].to_numpy(float)
        dof = valid["nis_dof"].to_numpy(int)

        # Normalized NIS
        nis_norm = raw / np.maximum(dof, 1)

        # DOF-aware threshold for normalized NIS: chi2.ppf(p, dof) / dof
        thr95_norm = chi2.ppf(0.95, dof) / dof
        thr99_norm = chi2.ppf(0.99, dof) / dof

        nis_rows.append({
            "model": model_name,
            "split": split_name,
            "median_nis_norm": float(np.nanmedian(nis_norm)),
            "mean_nis_norm": float(np.nanmean(nis_norm)),
            "p95_nis_norm": float(np.nanpercentile(nis_norm, 95)),
            "p99_nis_norm": float(np.nanpercentile(nis_norm, 99)),
            "pct_above_chi2_p95_norm": float(np.mean(nis_norm > thr95_norm) * 100),
            "pct_above_chi2_p99_norm": float(np.mean(nis_norm > thr99_norm) * 100),
            "n_samples": int(len(nis_norm)),
        })

nis_summary = pd.DataFrame(nis_rows)
nis_summary.to_csv(OUT_DIR / "nis_calibration_summary_dof_aware.csv", index=False)
print("\nNIS calibration summary (DOF-aware):")
print(nis_summary.to_string(index=False))

# WILCOXON TEST: KF VS ADAPTIVER (PAIRED OVER HIVES)

print("\nRunning Wilcoxon signed-rank test (KF vs AdaptiveR)...")

kf = (
    test_df[test_df["model"] == "KF"][["hive_id", "model_agg_zrmse"]]
    .rename(columns={"model_agg_zrmse": "kf"})
)
ar = (
    test_df[test_df["model"] == "AdaptiveR"][["hive_id", "model_agg_zrmse"]]
    .rename(columns={"model_agg_zrmse": "adaptiveR"})
)

paired = kf.merge(ar, on="hive_id", how="inner").dropna()

if len(paired) < 3:
    raise ValueError(f"Not enough paired hives for Wilcoxon (n={len(paired)}).")

stat, p_value = wilcoxon(paired["kf"].to_numpy(), paired["adaptiveR"].to_numpy())

diff = paired["kf"].to_numpy() - paired["adaptiveR"].to_numpy()  # positive => AdaptiveR lower error (better)
net_win_rate = float(np.mean(paired["adaptiveR"].to_numpy() < paired["kf"].to_numpy()))
median_delta = float(np.nanmedian(diff))
cd = float(cliff_delta(paired["kf"].to_numpy(), paired["adaptiveR"].to_numpy()))

# Bootstrap CI for the median paired difference
med, lo, hi = bootstrap_ci(diff, stat_fn=np.nanmedian, n_boot=4000)

print("Paired hives (n):", len(paired))
print("Wilcoxon statistic:", stat)
print("p-value:", p_value)
print("Win rate (AdaptiveR beats KF):", net_win_rate)
print("Median delta (kf - adaptiveR):", median_delta)
print("Cliff's delta (KF vs AdaptiveR):", cd)
print("Median delta bootstrap CI:", (med, lo, hi))

pd.DataFrame([{
    "comparison": "KF_vs_AdaptiveR",
    "wilcoxon_statistic": stat,
    "p_value": p_value,
    "n_hives_paired": int(len(paired)),
    "win_rate_adaptiveR_better": net_win_rate,
    "median_delta_kf_minus_adaptiveR": median_delta,
    "median_delta_ci_lo": lo,
    "median_delta_ci_hi": hi,
    "cliffs_delta_kf_vs_adaptiveR": cd,
}]).to_csv(OUT_DIR / "wilcoxon_kf_vs_adaptiveR_test.csv", index=False)

print("\nAll outputs written to:", OUT_DIR)
print("Figures directory:", FIG_DIR)
print("Tables directory :", TAB_DIR) 

TRAIN std: {'temperature': np.float64(2.0893432572964894), 'humidity': np.float64(6.593098136635135), 'audio_density': np.float64(7.089253512944294)}
Loading KF steps...
Loading EKF steps...
Loading EnKF steps...
Loading AdaptiveR steps...
Loading ARIMA steps...

Common hives (based on step availability):
  val : 26
  test: 26
Saved Table I outputs.

Baseline consistency check (test): max range across models = 0.0


C:\Users\DELL\AppData\Local\Temp\ipykernel_20992\3670376626.py:585: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  plt.boxplot(data, labels=labels, showfliers=False)


Saved: ..\backend\data\model_comparison\run_20260304_184215\figures\Figure1a_Test_Boxplot_LOG.png
Saved: ..\backend\data\model_comparison\run_20260304_184215\figures\Figure1a_Test_Boxplot_LOG.pdf


C:\Users\DELL\AppData\Local\Temp\ipykernel_20992\3670376626.py:605: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  plt.boxplot(data2, labels=labels2, showfliers=False)


Saved: ..\backend\data\model_comparison\run_20260304_184215\figures\Figure1b_Test_Boxplot_ZOOM.png
Saved: ..\backend\data\model_comparison\run_20260304_184215\figures\Figure1b_Test_Boxplot_ZOOM.pdf


C:\Users\DELL\AppData\Local\Temp\ipykernel_20992\3670376626.py:622: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  plt.boxplot(imp_data, labels=imp_labels, showfliers=False)


Saved: ..\backend\data\model_comparison\run_20260304_184215\figures\FigureA_ImprovementPct_Boxplot_Test.png
Saved: ..\backend\data\model_comparison\run_20260304_184215\figures\FigureA_ImprovementPct_Boxplot_Test.pdf
Saved: ..\backend\data\model_comparison\run_20260304_184215\figures\FigureB_KF_vs_Baseline_Scatter_Test.png
Saved: ..\backend\data\model_comparison\run_20260304_184215\figures\FigureB_KF_vs_Baseline_Scatter_Test.pdf
Saved: ..\backend\data\model_comparison\run_20260304_184215\figures\FigureB_AdaptiveR_vs_Baseline_Scatter_Test.png
Saved: ..\backend\data\model_comparison\run_20260304_184215\figures\FigureB_AdaptiveR_vs_Baseline_Scatter_Test.pdf
Saved: ..\backend\data\model_comparison\run_20260304_184215\figures\FigureC_PerHive_BestModel_Count_Test.png
Saved: ..\backend\data\model_comparison\run_20260304_184215\figures\FigureC_PerHive_BestModel_Count_Test.pdf
Saved: ..\backend\data\model_comparison\run_20260304_184215\figures\FigureD_MedianError_Ranking_Test.png
Saved: ..\backe

In [2]:
row_check = []
for model_name in MODELS:
    steps = steps_cache[model_name]
    df_s = steps[(steps["split"]=="test") & (steps["hive_id"].isin(common_test))]
    for hive_id, g in df_s.groupby("hive_id"):
        z = g[TRUE_COLS].to_numpy(float)
        p = g[PRED_COLS].to_numpy(float)
        valid = np.isfinite(z).any(axis=1) & np.isfinite(p).any(axis=1)
        row_check.append({
            "model": model_name,
            "hive_id": hive_id,
            "n_valid_rows": int(valid.sum())
        })

row_check_df = pd.DataFrame(row_check)
row_spread = row_check_df.groupby("hive_id")["n_valid_rows"].agg(["min","max"])
print("Max row-count difference across models:",
      (row_spread["max"] - row_spread["min"]).max())

Max row-count difference across models: 0
